# 🚀 Multi-Purpose Web Scraper Pipeline — Production Workflow

This notebook orchestrates a **production-grade web scraping pipeline** using the refactored, highly resilient `data_scraper_engine` interface.

### Dual Engine Capabilities:
1. **Google Play Store Reviews Scraper**: Direct API querying with SOTA rate-limiting mitigation (exponential backoff, dynamic jitter) and support for infinite/full review extraction.
2. **General CSS Web Scraper**: An asynchronous HTML scraper utilizing `scrapling` headless sessions to parse and extract items from dynamic sites.

All intermediate progress is saved to disk immediately via checkpoints, and progress is shown live on a premium Jupyter dashboard card.

---

---

### 📜 Unified Scraper Configuration Schema

The engine dynamically extracts and utilizes settings depending on the `scraper_type` value. Unused config keys are ignored.

| Group | Key | Type | Description | Default / Options | Example |
|:---|:---|:---|:---|:---|:---|
| **Global** | `scraper_type` | `str` | Orchestrator mode routing key. | `"google_play"`, `"html"` | `"google_play"` |
| **Global** | `output_filename` | `str` | Base path/filename on disk (no extension). | Any valid path stem | `"Exports/reviews_data"` |
| **Global** | `export_format` | `str` | consolidated file export format. | `"csv"`, `"jsonl"`, `"excel"` | `"csv"` |
| **Google Play** | `app_id` | `str` | Play Store App Package Identifier. | Play Store URL `id=` param | `"com.whatsapp"` |
| **Google Play** | `lang` | `str` | Review language target filter. | `"en"` | `"en"` |
| **Google Play** | `country` | `str` | Store region target filter. | `"us"` | `"us"` |
| **Google Play** | `sort` | `str` | Search results sorting ordering. | `"newest"`, `"most_relevant"` | `"newest"` |
| **Google Play** | `max_reviews` | `int`/`None` | Total count limit of reviews. | Set `None` for **ALL** reviews! | `500` |
| **Google Play** | `sleep_milliseconds`| `int` | Throttling wait delay between fetches. | `1000` | `1000` |
| **HTML / CSS** | `target_urls` | `List[str]` | List of web page URLs to scrape. | List of absolute URLs | `["https://quotes.toscrape.com/"]` |
| **HTML / CSS** | `container_selector` | `str` | Repeating parent DOM selector class/tag. | Any valid CSS selector | `".quote"` |
| **HTML / CSS** | `fields_to_scrape` | `Dict` | Map of column names to relative child selectors. | Dict of keys to CSS/method | `{"author": ".author"}` |

### 🚫 Pipeline Integrity Protections
1. **SOTA Resilience**: If Google Play Store returns 429 rate limit errors or network drops happen, the scraper auto-retries up to 5 times with exponential backoff delays.
2. **Dynamic Jitter**: Delay times are randomized around the `sleep_milliseconds` setting to avoid browser bot triggers.
3. **Automatic List Preservation**: Restore string-serialized lists back into Python lists when loading CSV checkpoints.

### 📋 Unified Blank Configuration Template

You can copy/paste this combined configuration ledger into the Step 1 code cell below. Toggle `scraper_type` between `"google_play"` and `"html"` to select the engine mode:

```python
SCRAPER_CONFIG = {
    # 1. GLOBAL PIPELINE CONTROLS
    "scraper_type": "google_play",            # Options: "google_play" or "html"
    "output_filename": "Exports/scraped_data",  # Output file base name (no extension)
    "export_format": "csv",                    # Options: "csv", "jsonl", "excel"
    
    # 2. 🤖 GOOGLE PLAY ENGINE SETTINGS (Used if scraper_type="google_play")
    "app_id": "com.whatsapp",                  # App Package ID (e.g. "com.whatsapp")
    "lang": "en",                              # Review language target filter
    "country": "us",                           # Store region country filter
    "sort": "newest",                          # Options: "newest", "most_relevant"
    "max_reviews": None,                       # Set to None for ALL reviews
    "sleep_milliseconds": 1000,                # Delay between fetches
    
    # 3. 🕷️ GENERAL HTML CSS ENGINE SETTINGS (Used if scraper_type="html")
    "target_urls": [
        "https://quotes.toscrape.com/page/1/",
        "https://quotes.toscrape.com/page/2/"
    ],
    "container_selector": ".quote",
    "fields_to_scrape": {
        "author": {"selector": ".author::text", "method": "get"},
        "quote_text": {"selector": ".text::text", "method": "get"},
        "tags": {"selector": ".tags .tag::text", "method": "getall"}
    }
}
```

---
## ⚙️ Step 1: Web Scraper Configuration
Configure your scraper parameters using the combined configuration dictionary below. Change `scraper_type` to switch modes seamlessly.

In [1]:
# ══════════════════════════════════════════════════════════════════════
#  Unified Pipeline Configuration Ledger
# ══════════════════════════════════════════════════════════════════════

SCRAPER_CONFIG = {
    # 1. GLOBAL PIPELINE CONTROLS
    "scraper_type": "google_play",            # Options: "google_play" or "html"
    "output_filename": "Exports/scraped_data",  # Output file base name (no extension)
    "export_format": "csv",                    # Options: "csv", "jsonl", "excel"
    
    # 2. 🤖 GOOGLE PLAY ENGINE SETTINGS (Used if scraper_type="google_play")
    "app_id": "id.astra.dso.digitalization",   # App Package ID (e.g. "com.whatsapp")
    "lang": "id",                              # Review language target filter
    "country": "id",                           # Store region country filter
    "sort": "newest",                          # Options: "newest", "most_relevant"
    "max_reviews": None,                       # Set to None for ALL reviews
    "sleep_milliseconds": 1000,                # Delay between fetches
    
    # 3. 🕷️ GENERAL HTML CSS ENGINE SETTINGS (Used if scraper_type="html")
    "target_urls": [
        "https://quotes.toscrape.com/page/1/",
        "https://quotes.toscrape.com/page/2/"
    ],
    "container_selector": ".quote",
    "fields_to_scrape": {
        "author": {"selector": ".author::text", "method": "get"},
        "quote_text": {"selector": ".text::text", "method": "get"},
        "tags": {"selector": ".tags .tag::text", "method": "getall"}
    }
}

---
## 🕷️ Step 2: Web Scraper Pipeline Execution
Running this block initiates the isolated scraper process. Progress will be displayed live on the dashboard card, and data will be saved continuously to the checkpoint file.

In [2]:
# ══════════════════════════════════════════════════════════════════════
#  Pipeline Execution & Consolidation
# ══════════════════════════════════════════════════════════════════════

from data_scraper_engine import execute_pipeline
df = execute_pipeline(SCRAPER_CONFIG)

review_id,user_name,user_image_url,review_text,rating,likes,review_app_version,review_date,developer_reply,reply_date,app_version
1c210513-3fc5-4ced-aa44-f263bf25d020,Bang udding Udding,https://play-lh.googleusercontent.com/a/ACg8ocJljf7JDGxkvgW715bRhvdWSmJQABoFGHb-tVPoSVcEHt9TKA=mo,sangat memuaskan,5,0,2.30.0,2026-05-12 06:23:01,NaN,NaN,2.30.0
b34af1e1-63ba-4929-beca-95da8e99d784,Fendi Juniariandi,https://play-lh.googleusercontent.com/a-/ALV-UjWx1QlyIZGoYVEeTss5hpd0565iCbro8YQw7cEqfKk2MBHRTLQ,"Pelayanan yang sangat parah kejadian di Tunas Daihatsu Palembang keluhan kita tidak ada yang diselesaikan pihak Bengkel, kita minta cek ulang malah di minta biaya tambahan",1,0,NaN,2026-05-10 08:58:30,NaN,NaN,NaN
5e32b35d-f633-4d2b-b329-ed43cbf3fea1,Bima Marìtza,https://play-lh.googleusercontent.com/a-/ALV-UjWMiwHUAFVHpAwRC_e4_smVXx1rahv1oUIOYzC_CoU3gi9rh9DE,Amat sangat berguna!,5,0,2.30.0,2026-04-29 16:43:47,NaN,NaN,2.30.0
14dfbd0e-374d-4c4e-a6d7-8f6997f761c5,Arvinza Risfa ramadhan,https://play-lh.googleusercontent.com/a/ACg8ocJLv0492tMZK4p3mUL9aB_EP026zU7A8b5d9m2-7Z_X5MWELbM=mo,aplikasi nya gak update kah?,5,0,NaN,2026-04-21 06:51:53,NaN,NaN,NaN
1fc09323-6934-458a-9c49-da9e55b247e8,Hadrian Hartoko,https://play-lh.googleusercontent.com/a-/ALV-UjU1XiDmqy5qdYi0Jx-M8dQJqZIpETldi8l2BwATF2cF6jkrG8wp6w,data riwayat servis tidak muncul padahal servisnya di bengkel resmi,1,0,2.30.0,2026-04-18 10:20:09,NaN,NaN,2.30.0


---
## 🛠️ Step-by-Step Guide: How to Configure the Scraper

### Mode A: Google Play Reviews Scraper
1. **Find the App ID**: Go to the Google Play Store on your browser, search for the app, and look at the URL. Copy the part after `id=` (e.g., `com.whatsapp`, `com.spotify.music`).
2. **Set Configuration**: Set `scraper_type` to `"google_play"` and pass the `app_id`.
3. **Customize Filters**:
   - `lang`: Language code (e.g., `"en"`, `"id"`).
   - `country`: Country store (e.g., `"us"`, `"id"`).
   - `sort`: Set to `"newest"` (for tracking fresh feedback) or `"most_relevant"`.
   - `max_reviews`: Target quantity of reviews. Set to `None` to fetch **ALL** available reviews.
4. **Run**: Execute Step 1 and Step 2 cells. The interactive HTML console will display your progress live.

### Mode B: General HTML CSS Scraper
1. **Identify Repeating Containers**: Open the target site in your browser. Inspect (`F12`) to find the repeating CSS selector wrapping each item (e.g., `".quote"`, `"div.product-card"`). This is your `container_selector`.
2. **Map Child Selectors**: Inspect elements inside the container and get their relative selectors (e.g., `.author::text`, `span.price::text`).
3. **Map Extraction Types**:
   - Single item: `"method": "get"`
   - Multiple items (list): `"method": "getall"`
4. **Set Configuration**: Set `scraper_type` to `"html"`, configure your selectors, and run!